In [ ]:
# Create session 
from snowflake.snowpark import Session
session = Session.builder.getOrCreate()

# Attach warehouse
session.sql("USE WAREHOUSE CHIPMUNK_WH").collect()

# Verify
session.sql("SELECT CURRENT_WAREHOUSE()").show()

# Test data access
session.sql("SELECT COUNT(*) FROM ALL_TRIPS").show()

# See all features available
session.sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'ALL_TRIPS'
    ORDER BY ordinal_position
""").show()

# View first 10 rows
session.sql("""
    SELECT *
    FROM ALL_TRIPS
    LIMIT 10
""").show()

# Aggregate data at monthly level
monthly = session.sql("""
    SELECT
        DATE_TRUNC('month', trip_date) AS month,
        service_type,
        COUNT(*) AS trips
    FROM ALL_TRIPS
    GROUP BY month, service_type
    ORDER BY month
""")

# Convert to pandas
monthly_df = monthly.to_pandas()
monthly_df.head()

# Some cells have corrupted data; check which ones
session.sql("""
    SELECT MIN(trip_date), MAX(trip_date)
    FROM ALL_TRIPS
""").show()

# Fix monthly aggregation
monthly = session.sql("""
    SELECT
        DATE_TRUNC('month', trip_date) AS month,
        service_type,
        COUNT(*) AS trips
    FROM ALL_TRIPS
    WHERE trip_date >= '2024-01-01'
      AND trip_date <= '2026-06-30'
    GROUP BY month, service_type
    ORDER BY month
""")

# check monthly data
monthly_df = monthly.to_pandas()
monthly_df.head()

# Ensure inline plotting (important in Snowflake notebooks)
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd

# --- Fix datetime issues ---
monthly_df["MONTH"] = pd.to_datetime(monthly_df["MONTH"])
monthly_df = monthly_df.sort_values("MONTH")

# Policy date
policy_date = pd.to_datetime("2025-01-01")

# --- Plot ---
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd

# Fix datetime
monthly_df["MONTH"] = pd.to_datetime(monthly_df["MONTH"])
monthly_df = monthly_df.sort_values("MONTH")

policy_date = pd.to_datetime("2025-01-01")

plt.figure(figsize=(10, 6))

for service in monthly_df["SERVICE_TYPE"].unique():
    subset = monthly_df[monthly_df["SERVICE_TYPE"] == service]
    plt.plot(subset["MONTH"], subset["TRIPS"], label=service)

plt.axvline(x=policy_date, linestyle="--", color="black", label="Policy Start")

plt.xlabel("Month")
plt.ylabel("Number of Trips")
plt.title("Monthly Trip Volume by Service Type")
plt.legend()

plt.xticks(rotation=45)

plt.tight_layout()

# FORCE render
display(plt.gcf())
plt.close()

# Create weekly panel
weekly = session.sql("""
    SELECT
        DATE_TRUNC('week', trip_date) AS week,
        PULocationID,
        COUNT(*) AS trips
    FROM ALL_TRIPS
    WHERE trip_date >= '2024-01-01'
""")

# Check if CBD Congestion Fee is only implemented after the policy date
session.sql("""
    SELECT
        DATE_TRUNC('week', trip_date) AS week,
        AVG(cbd_congestion_fee) AS avg_fee
    FROM ALL_TRIPS
    WHERE trip_date >= '2024-01-01'
    GROUP BY week
    ORDER BY week
""").show()

# Build weekly data panel
weekly = session.sql("""
    SELECT
        DATE_TRUNC('week', trip_date) AS week,
        PULocationID,
        service_type,
        COUNT(*) AS trips
    FROM ALL_TRIPS
    WHERE trip_date >= '2024-01-01'
    GROUP BY week, PULocationID, service_type
""")

# Convert to pandas
weekly_df = weekly.to_pandas

# Fix data types
import pandas as pd
import numpy as np

weekly_df["WEEK"] = pd.to_datetime(weekly_df["WEEK"])

# Define Manhattan tax unit location IDs
manhattan_ids = [
    4,12,13,24,41,42,43,48,50,68,74,75,79,87,88,90,
    100,103,104,105,107,113,114,116,120,125,137,140,
    141,142,143,144,148,151,152,158,161,162,163,164,
    166,170,186,194,202,209,211,224,229,230,231,232,
    233,234,236,237,238,239,243,244,246,249,261,262,263
]

weekly_df["treated"] = weekly_df["PULOCATIONID"].isin(manhattan_ids).astype(int)

# Add treatment + post
weekly_df["treated"] = weekly_df["cbd_congestion_fee"].isin(manhattan_ids).astype(int)
weekly_df["post"] = (weekly_df["WEEK"] >= "2025-01-05").astype(int)

# Create outcome variable (log of # trips)
import numpy as np
weekly_df["log_trips"] = np.log1p(weekly_df["TRIPS"])

# Run baseline DiD model
import statsmodels.formula.api as smf

model = smf.ols(
    "log_trips ~ treated * post + C(PULOCATIONID) + C(WEEK)",
    data=weekly_df
).fit()

print(model.summary())

# Get relevant results
print(model.params["treated:post"])
print(model.pvalues["treated:post"])

# Print relevant results
import pandas as pd
import numpy as np

# Extract key results
results = pd.DataFrame({
    "Coefficient": model.params,
    "Std_Error": model.bse,
    "P_value": model.pvalues
})

# Keep only relevant variables
results_clean = results.loc[[
    "treated:post",
    "treated",
    "post"
]]

# Add % effect column (for interpretation)
results_clean["Pct_Effect"] = (np.exp(results_clean["Coefficient"]) - 1) * 100

print(results_clean)

# Print clean results
results_clean_rounded = results_clean.copy()

results_clean_rounded["Coefficient"] = results_clean_rounded["Coefficient"].round(3)
results_clean_rounded["Std_Error"] = results_clean_rounded["Std_Error"].round(3)
results_clean_rounded["Pct_Effect"] = results_clean_rounded["Pct_Effect"].round(1)

print(results_clean_rounded)

# Visualize results
# Create event time
weekly_df["event_time"] = (
    (weekly_df["WEEK"] - pd.to_datetime("2025-01-05"))
    .dt.days // 7
)

event = weekly_df.groupby(["event_time", "treated"])["log_trips"].mean().reset_index()

plt.figure(figsize=(10,6))

for t in [0, 1]:
    subset = event[event["treated"] == t]
    label = "Treated" if t == 1 else "Control"
    plt.plot(subset["event_time"], subset["log_trips"], label=label)

plt.axvline(0, linestyle="--", color="black")

plt.title("Event Study: Trip Volume Around Policy")
plt.xlabel("Weeks Relative to Policy")
plt.ylabel("Log Trips")
plt.legend()

display(plt.gcf())
plt.close()